# Week 3 – KPI Analysis

## Objective

The purpose of this notebook is to calculate and consolidate key business KPIs using the validated and feature-engineered datasets.

The KPI analysis covers:

- Sales performance
- Order performance
- Product performance
- Customer performance
- Supplier coverage
- Inventory levels
- Stock health
- Stock utilisation
- Reorder requirements
- Branch performance
- Sales channel performance

In [1]:
import pandas as pd
import numpy as np

pd.set_option(
    'display.max_columns',
    None
)

In [13]:
sales = pd.read_csv(
    "/content/sales_features.csv"
)

products = pd.read_csv(
    "/content/products.csv"
)

customers = pd.read_csv(
    "/content/customers.csv"
)

suppliers = pd.read_csv(
    "/content/suppliers.csv"
)

inventory = pd.read_csv(
    "/content/inventory_master.csv"
)

purchase_integrated = pd.read_csv(
    "/content/purchase_integrated.csv"
)


print("Sales:", sales.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Suppliers:", suppliers.shape)
print("Inventory:", inventory.shape)
print("Purchase Integrated:", purchase_integrated.shape)

Sales: (130402, 83)
Products: (30, 23)
Customers: (500, 15)
Suppliers: (8, 12)
Inventory: (180, 8)
Purchase Integrated: (155495, 51)


Check Required Sales Columns

In [11]:
sales_columns = [
    'so_id',
    'product_id',
    'customer_id',
    'quantity',
    'line_grand_total'
]

print("Sales column check:")

for col in sales_columns:

    print(
        f"{col}:",
        "FOUND" if col in sales.columns else "NOT FOUND"
    )

Sales column check:
so_id: FOUND
product_id: FOUND
customer_id: FOUND
quantity: FOUND
line_grand_total: FOUND


Check Inventory Columns

In [14]:
inventory_columns = [
    'product_id',
    'branch_id',
    'opening_stock',
    'reorder_level',
    'safety_stock',
    'max_stock',
    'current_stock',
    'warehouse_bin'
]

print("Inventory column check:")

for col in inventory_columns:

    print(
        f"{col}:",
        "FOUND" if col in inventory.columns else "NOT FOUND"
    )

Inventory column check:
product_id: FOUND
branch_id: FOUND
opening_stock: FOUND
reorder_level: FOUND
safety_stock: FOUND
max_stock: FOUND
current_stock: FOUND
warehouse_bin: FOUND


Basic Sales KPI

In [16]:
total_sales = (
    sales['line_grand_total'].sum()
)

total_orders = (
    sales['so_id'].nunique()
)

total_quantity_sold = (
    sales['quantity'].sum()
)

total_products = (
    sales['product_id'].nunique()
)

total_customers = (
    sales['customer_id'].nunique()
)

average_order_value = (
    total_sales / total_orders
    if total_orders > 0
    else 0
)

average_quantity_per_order = (
    total_quantity_sold / total_orders
    if total_orders > 0
    else 0
)


# Display Sales KPI
print("=" * 60)
print("SALES KPIs")
print("=" * 60)

print(
    "Total Sales:",
    f"{total_sales:,.2f}"
)

print(
    "Total Orders:",
    f"{total_orders:,}"
)

print(
    "Total Quantity Sold:",
    f"{total_quantity_sold:,.0f}"
)

print(
    "Total Products:",
    f"{total_products:,}"
)

print(
    "Total Customers:",
    f"{total_customers:,}"
)

print(
    "Average Order Value:",
    f"{average_order_value:,.2f}"
)

print(
    "Average Quantity per Order:",
    f"{average_quantity_per_order:,.2f}"
)

SALES KPIs
Total Sales: 32,424,754,555.80
Total Orders: 20,000
Total Quantity Sold: 1,368,534
Total Products: 30
Total Customers: 500
Average Order Value: 1,621,237.73
Average Quantity per Order: 68.43


Product KPIs

In [17]:
product_kpi = (
    sales
    .groupby('product_id')
    .agg(
        total_sales=(
            'line_grand_total',
            'sum'
        ),
        total_quantity_sold=(
            'quantity',
            'sum'
        ),
        total_orders=(
            'so_id',
            'nunique'
        ),
        total_customers=(
            'customer_id',
            'nunique'
        )
    )
    .reset_index()
)

product_kpi['average_sales_per_order'] = np.where(
    product_kpi['total_orders'] > 0,
    product_kpi['total_sales']
    / product_kpi['total_orders'],
    0
)

display(
    product_kpi
    .sort_values(
        'total_sales',
        ascending=False
    )
    .head(10)
)

,product_id,total_sales,total_quantity_sold,total_orders,total_customers,average_sales_per_order
14,P015,6.888215e+09,45222,3773,499,1.825660e+06
13,P014,4.813341e+09,45035,3838,500,1.254127e+06
20,P021,2.741782e+09,46364,3873,500,7.079219e+05
18,P019,2.341947e+09,45971,3849,499,6.084559e+05
3,P004,2.321773e+09,45575,3781,500,6.140632e+05
9,P010,2.097976e+09,46964,3933,500,5.334289e+05
10,P011,1.646709e+09,46612,3982,500,4.135381e+05
0,P001,1.423399e+09,45389,3854,500,3.693303e+05
29,P030,1.286585e+09,45897,3856,500,3.336579e+05
21,P022,9.911566e+08,45819,3892,500,2.546651e+05


Top Products

In [18]:
top_product = (
    product_kpi
    .sort_values(
        'total_sales',
        ascending=False
    )
    .iloc[0]
)

print("Top Product:")
display(
    top_product.to_frame(
        name='Value'
    )
)

Top Product:


,Value
product_id,P015
total_sales,6888215040.0
total_quantity_sold,45222
total_orders,3773
total_customers,499
average_sales_per_order,1825659.962894


Customers KPIs

In [19]:
customer_kpi = (
    sales
    .groupby('customer_id')
    .agg(
        total_sales=(
            'line_grand_total',
            'sum'
        ),
        total_orders=(
            'so_id',
            'nunique'
        ),
        total_quantity=(
            'quantity',
            'sum'
        ),
        unique_products=(
            'product_id',
            'nunique'
        )
    )
    .reset_index()
)

customer_kpi['average_order_value'] = np.where(
    customer_kpi['total_orders'] > 0,
    customer_kpi['total_sales']
    / customer_kpi['total_orders'],
    0
)

display(
    customer_kpi
    .sort_values(
        'total_sales',
        ascending=False
    )
    .head(10)
)

,customer_id,total_sales,total_orders,total_quantity,unique_products,average_order_value
152,C0153,109360638.6,59,4256,30,1.853570e+06
482,C0483,108672502.0,53,3846,30,2.050425e+06
287,C0288,107119652.2,63,4626,30,1.700312e+06
494,C0495,105717422.2,52,3819,30,2.033027e+06
419,C0420,105420085.2,54,4133,30,1.952224e+06
239,C0240,104286776.8,50,3799,30,2.085736e+06
206,C0207,101737768.4,51,3583,30,1.994858e+06
341,C0342,100153639.6,44,3725,30,2.276219e+06
465,C0466,99913868.2,52,3880,30,1.921421e+06
358,C0359,99895983.4,52,3851,30,1.921077e+06


Top Customers

In [20]:
top_customer = (
    customer_kpi
    .sort_values(
        'total_sales',
        ascending=False
    )
    .iloc[0]
)

print("Top Customer:")
display(
    top_customer.to_frame(
        name='Value'
    )
)

Top Customer:


,Value
customer_id,C0153
total_sales,109360638.6
total_orders,59
total_quantity,4256
unique_products,30
average_order_value,1853570.145763


Supplier KPIs

In [21]:
if 'supplier_id' in suppliers.columns:

    total_suppliers = (
        suppliers['supplier_id']
        .nunique()
    )

else:

    total_suppliers = len(
        suppliers.drop_duplicates()
    )

print(
    "Total Suppliers:",
    f"{total_suppliers:,}"
)

Total Suppliers: 8


Inventory KPIs

In [22]:
# ==========================================
# INVENTORY KPIs
# ==========================================

total_inventory_records = (
    len(inventory)
)

total_inventory_products = (
    inventory['product_id'].nunique()
)

total_inventory_branches = (
    inventory['branch_id'].nunique()
)

total_opening_stock = (
    inventory['opening_stock'].sum()
)

total_current_stock = (
    inventory['current_stock'].sum()
)

total_reorder_level = (
    inventory['reorder_level'].sum()
)

total_safety_stock = (
    inventory['safety_stock'].sum()
)

total_max_stock = (
    inventory['max_stock'].sum()
)

print("=" * 60)
print("INVENTORY KPIs")
print("=" * 60)

print(
    "Total Inventory Records:",
    f"{total_inventory_records:,}"
)

print(
    "Total Products in Inventory:",
    f"{total_inventory_products:,}"
)

print(
    "Total Branches with Inventory:",
    f"{total_inventory_branches:,}"
)

print(
    "Total Opening Stock:",
    f"{total_opening_stock:,.0f}"
)

print(
    "Total Current Stock:",
    f"{total_current_stock:,.0f}"
)

print(
    "Total Reorder Level:",
    f"{total_reorder_level:,.0f}"
)

print(
    "Total Safety Stock:",
    f"{total_safety_stock:,.0f}"
)

print(
    "Total Maximum Stock Capacity:",
    f"{total_max_stock:,.0f}"
)

INVENTORY KPIs
Total Inventory Records: 180
Total Products in Inventory: 30
Total Branches with Inventory: 6
Total Opening Stock: 32,969
Total Current Stock: 19,303,266
Total Reorder Level: 11,619
Total Safety Stock: 8,137
Total Maximum Stock Capacity: 57,916


Stock Health KPIs

In [23]:
# ==========================================
# STOCK HEALTH KPIs
# ==========================================

out_of_stock_count = (
    inventory['current_stock'] <= 0
).sum()

below_reorder_count = (
    inventory['current_stock']
    < inventory['reorder_level']
).sum()

below_safety_count = (
    inventory['current_stock']
    < inventory['safety_stock']
).sum()

above_max_stock_count = (
    inventory['current_stock']
    > inventory['max_stock']
).sum()

print("=" * 60)
print("STOCK HEALTH KPIs")
print("=" * 60)

print(
    "Out-of-Stock Records:",
    f"{out_of_stock_count:,}"
)

print(
    "Below Reorder Level:",
    f"{below_reorder_count:,}"
)

print(
    "Below Safety Stock:",
    f"{below_safety_count:,}"
)

print(
    "Above Maximum Stock:",
    f"{above_max_stock_count:,}"
)

STOCK HEALTH KPIs
Out-of-Stock Records: 0
Below Reorder Level: 0
Below Safety Stock: 0
Above Maximum Stock: 180


Stock Utilisation KPI

In [24]:
inventory['stock_utilisation_pct'] = np.where(
    inventory['max_stock'] > 0,
    (
        inventory['current_stock']
        / inventory['max_stock']
    ) * 100,
    np.nan
)

average_stock_utilisation = (
    inventory['stock_utilisation_pct']
    .mean()
)

print(
    "Average Stock Utilisation:",
    f"{average_stock_utilisation:.2f}%"
)

Average Stock Utilisation: 38503.99%


Recorder Gap KPI

In [25]:
inventory['reorder_gap'] = (
    inventory['reorder_level']
    - inventory['current_stock']
).clip(lower=0)

total_reorder_gap = (
    inventory['reorder_gap'].sum()
)

print(
    "Total Reorder Gap:",
    f"{total_reorder_gap:,.0f}"
)

Total Reorder Gap: 0


Inventory KPI Table

In [26]:
inventory_kpis = {
    'Total Inventory Records':
        total_inventory_records,

    'Total Products in Inventory':
        total_inventory_products,

    'Total Branches with Inventory':
        total_inventory_branches,

    'Total Opening Stock':
        total_opening_stock,

    'Total Current Stock':
        total_current_stock,

    'Total Reorder Level':
        total_reorder_level,

    'Total Safety Stock':
        total_safety_stock,

    'Total Maximum Stock Capacity':
        total_max_stock,

    'Out-of-Stock Records':
        out_of_stock_count,

    'Below Reorder Level':
        below_reorder_count,

    'Below Safety Stock':
        below_safety_count,

    'Above Maximum Stock':
        above_max_stock_count,

    'Average Stock Utilisation (%)':
        average_stock_utilisation,

    'Total Reorder Gap':
        total_reorder_gap
}

inventory_kpi_df = pd.DataFrame(
    inventory_kpis.items(),
    columns=[
        'KPI',
        'Value'
    ]
)

display(inventory_kpi_df)

,KPI,Value
0,Total Inventory Records,1.800000e+02
1,Total Products in Inventory,3.000000e+01
2,Total Branches with Inventory,6.000000e+00
3,Total Opening Stock,3.296900e+04
4,Total Current Stock,1.930327e+07
5,Total Reorder Level,1.161900e+04
6,Total Safety Stock,8.137000e+03
7,Total Maximum Stock Capacity,5.791600e+04
8,Out-of-Stock Records,0.000000e+00
9,Below Reorder Level,0.000000e+00


Sales Channel KPIs

In [27]:
if 'sales_channel' in sales.columns:

    channel_kpi = (
        sales
        .groupby('sales_channel')
        .agg(
            total_sales=(
                'line_grand_total',
                'sum'
            ),
            total_orders=(
                'so_id',
                'nunique'
            ),
            total_quantity=(
                'quantity',
                'sum'
            ),
            total_customers=(
                'customer_id',
                'nunique'
            )
        )
        .reset_index()
    )

    channel_kpi['average_order_value'] = np.where(
        channel_kpi['total_orders'] > 0,
        channel_kpi['total_sales']
        / channel_kpi['total_orders'],
        0
    )

    display(
        channel_kpi
        .sort_values(
            'total_sales',
            ascending=False
        )
    )

,sales_channel,total_sales,total_orders,total_quantity,total_customers,average_order_value
2,Field Sales,8.210171e+09,5036,346107,500,1.630296e+06
0,Counter Sale,8.172855e+09,5014,345148,500,1.630007e+06
1,Dealer Network,8.107783e+09,5028,342006,500,1.612527e+06
3,Online,7.933945e+09,4922,335273,500,1.611935e+06


Brach KPIs

In [28]:
if 'branch_id' in sales.columns:

    branch_kpi = (
        sales
        .groupby('branch_id')
        .agg(
            total_sales=(
                'line_grand_total',
                'sum'
            ),
            total_orders=(
                'so_id',
                'nunique'
            ),
            total_quantity=(
                'quantity',
                'sum'
            ),
            total_customers=(
                'customer_id',
                'nunique'
            )
        )
        .reset_index()
    )

    branch_kpi['average_order_value'] = np.where(
        branch_kpi['total_orders'] > 0,
        branch_kpi['total_sales']
        / branch_kpi['total_orders'],
        0
    )

    branch_kpi = branch_kpi.sort_values(
        'total_sales',
        ascending=False
    )

    display(branch_kpi)

,branch_id,total_sales,total_orders,total_quantity,total_customers,average_order_value
1,CHN001,6.615942e+09,4067,280446,102,1.626738e+06
4,KOL001,6.100494e+09,3794,256095,96,1.607932e+06
0,AHM001,5.616586e+09,3446,236347,88,1.629886e+06
2,DEL001,4.926515e+09,2953,204734,71,1.668308e+06
5,PUN001,4.766708e+09,2972,201749,75,1.603872e+06
3,HYD001,4.398510e+09,2768,189163,68,1.589057e+06


Purchase KPIs

In [30]:
if purchase_integrated is not None:

    print(
        "Purchase columns:"
    )

    print(
        purchase_integrated.columns.tolist()
    )

Purchase columns:
['po_id', 'line_number', 'product_id', 'quantity', 'unit_cost', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total', 'supplier_id', 'branch_id', 'order_date', 'expected_delivery_date', 'received_date', 'po_status', 'total_cost', 'total_gst_amount', 'grand_total', 'supplier_name', 'supplier_type', 'product_category', 'city', 'province', 'region', 'pincode', 'lead_time_days', 'reliability_score', 'import_duty_rate', 'china_tax_id', 'product_name', 'category', 'machine_type', 'brand', 'model_compatibility', 'unit_cost_product', 'unit_price', 'margin_percentage', 'gst_rate_product', 'weight_kg', 'dimensions_cm', 'material_type', 'warranty_months', 'reorder_level', 'safety_stock', 'max_stock_level', 'lead_time_days_product', 'criticality_level', 'usage_frequency', 'uom', 'last_purchase_price', 'last_purchase_date']


Detect Purchase Value Columns

In [32]:
if purchase_integrated is not None:

    possible_purchase_columns = [
        col
        for col in purchase_integrated.columns
        if any(
            word in col.lower()
            for word in [
                'grand_total',
                'total_value',
                'total_amount',
                'order_value'
            ]
        )
    ]

    print(
        "Possible purchase value columns:"
    )

    print(
        possible_purchase_columns
    )

Possible purchase value columns:
['line_grand_total', 'grand_total']


In [34]:
if (
    purchase_integrated is not None
    and 'grand_total' in purchase_integrated.columns
):

    total_purchase_value = (
        purchase_integrated['grand_total'].sum()
    )

    print(
        "Total Purchase Value:",
        f"{total_purchase_value:,.2f}"
    )

Total Purchase Value: 3,681,014,730,852.53


Overall KPI Table

In [36]:
overall_kpis = {
    'Total Sales':
        total_sales,

    'Total Orders':
        total_orders,

    'Total Quantity Sold':
        total_quantity_sold,

    'Total Products':
        total_products,

    'Total Customers':
        total_customers,

    'Total Suppliers':
        total_suppliers,

    'Average Order Value':
        average_order_value,

    'Average Quantity per Order':
        average_quantity_per_order
}

if purchase_integrated is not None:

    if 'grand_total' in purchase_integrated.columns:

        overall_kpis[
            'Total Purchase Value'
        ] = total_purchase_value

overall_kpi_df = pd.DataFrame(
    overall_kpis.items(),
    columns=[
        'KPI',
        'Value'
    ]
)

display(overall_kpi_df)

,KPI,Value
0,Total Sales,3.242475e+10
1,Total Orders,2.000000e+04
2,Total Quantity Sold,1.368534e+06
3,Total Products,3.000000e+01
4,Total Customers,5.000000e+02
5,Total Suppliers,8.000000e+00
6,Average Order Value,1.621238e+06
7,Average Quantity per Order,6.842670e+01
8,Total Purchase Value,3.681015e+12


Combine All KPIs

In [37]:
kpi_df = pd.concat(
    [
        overall_kpi_df,
        inventory_kpi_df
    ],
    ignore_index=True
)

display(kpi_df)

,KPI,Value
0,Total Sales,3.242475e+10
1,Total Orders,2.000000e+04
2,Total Quantity Sold,1.368534e+06
3,Total Products,3.000000e+01
4,Total Customers,5.000000e+02
5,Total Suppliers,8.000000e+00
6,Average Order Value,1.621238e+06
7,Average Quantity per Order,6.842670e+01
8,Total Purchase Value,3.681015e+12
9,Total Inventory Records,1.800000e+02


Top Product KPI

In [38]:
top_product = (
    product_kpi
    .sort_values(
        'total_sales',
        ascending=False
    )
    .iloc[0]
)

print(
    "Top Product by Sales:",
    top_product['product_id']
)

print(
    "Sales:",
    f"{top_product['total_sales']:,.2f}"
)

print(
    "Quantity Sold:",
    f"{top_product['total_quantity_sold']:,.0f}"
)

Top Product by Sales: P015
Sales: 6,888,215,040.00
Quantity Sold: 45,222


Top Customer KPI

In [39]:
top_customer = (
    customer_kpi
    .sort_values(
        'total_sales',
        ascending=False
    )
    .iloc[0]
)

print(
    "Top Customer:",
    top_customer['customer_id']
)

print(
    "Sales:",
    f"{top_customer['total_sales']:,.2f}"
)

print(
    "Orders:",
    f"{top_customer['total_orders']:,}"
)

Top Customer: C0153
Sales: 109,360,638.60
Orders: 59


Inventory Risk Summary

In [40]:
print("=" * 60)
print("INVENTORY RISK SUMMARY")
print("=" * 60)

print(
    "Out of Stock:",
    f"{out_of_stock_count:,}"
)

print(
    "Below Reorder Level:",
    f"{below_reorder_count:,}"
)

print(
    "Below Safety Stock:",
    f"{below_safety_count:,}"
)

print(
    "Above Maximum Stock:",
    f"{above_max_stock_count:,}"
)

print(
    "Average Stock Utilisation:",
    f"{average_stock_utilisation:.2f}%"
)

print(
    "Total Reorder Gap:",
    f"{total_reorder_gap:,.0f}"
)

INVENTORY RISK SUMMARY
Out of Stock: 0
Below Reorder Level: 0
Below Safety Stock: 0
Above Maximum Stock: 180
Average Stock Utilisation: 38503.99%
Total Reorder Gap: 0


Product Performance Summary

In [41]:
print("=" * 60)
print("PRODUCT PERFORMANCE")
print("=" * 60)

print("\nTop 10 Products by Sales:")

display(
    product_kpi
    .sort_values(
        'total_sales',
        ascending=False
    )
    .head(10)
)

print("\nTop 10 Products by Quantity Sold:")

display(
    product_kpi
    .sort_values(
        'total_quantity_sold',
        ascending=False
    )
    .head(10)
)

PRODUCT PERFORMANCE

Top 10 Products by Sales:


,product_id,total_sales,total_quantity_sold,total_orders,total_customers,average_sales_per_order
14,P015,6.888215e+09,45222,3773,499,1.825660e+06
13,P014,4.813341e+09,45035,3838,500,1.254127e+06
20,P021,2.741782e+09,46364,3873,500,7.079219e+05
18,P019,2.341947e+09,45971,3849,499,6.084559e+05
3,P004,2.321773e+09,45575,3781,500,6.140632e+05
9,P010,2.097976e+09,46964,3933,500,5.334289e+05
10,P011,1.646709e+09,46612,3982,500,4.135381e+05
0,P001,1.423399e+09,45389,3854,500,3.693303e+05
29,P030,1.286585e+09,45897,3856,500,3.336579e+05
21,P022,9.911566e+08,45819,3892,500,2.546651e+05



Top 10 Products by Quantity Sold:


,product_id,total_sales,total_quantity_sold,total_orders,total_customers,average_sales_per_order
28,P029,2.193746e+08,47066,3882,500,56510.722823
9,P010,2.097976e+09,46964,3933,500,533428.885838
5,P006,6.699356e+08,46731,3936,500,170207.219512
10,P011,1.646709e+09,46612,3982,500,413538.105475
6,P007,7.950675e+07,46468,3906,499,20355.030210
20,P021,2.741782e+09,46364,3873,500,707921.896204
15,P016,5.573390e+08,46306,3934,500,141672.347738
27,P028,1.225654e+08,46164,3917,500,31290.635691
11,P012,1.441682e+08,45972,3883,500,37128.043266
18,P019,2.341947e+09,45971,3849,499,608455.864900


Customer Performance Summary

In [42]:
print("=" * 60)
print("CUSTOMER PERFORMANCE")
print("=" * 60)

display(
    customer_kpi
    .sort_values(
        'total_sales',
        ascending=False
    )
    .head(10)
)

CUSTOMER PERFORMANCE


,customer_id,total_sales,total_orders,total_quantity,unique_products,average_order_value
152,C0153,109360638.6,59,4256,30,1.853570e+06
482,C0483,108672502.0,53,3846,30,2.050425e+06
287,C0288,107119652.2,63,4626,30,1.700312e+06
494,C0495,105717422.2,52,3819,30,2.033027e+06
419,C0420,105420085.2,54,4133,30,1.952224e+06
239,C0240,104286776.8,50,3799,30,2.085736e+06
206,C0207,101737768.4,51,3583,30,1.994858e+06
341,C0342,100153639.6,44,3725,30,2.276219e+06
465,C0466,99913868.2,52,3880,30,1.921421e+06
358,C0359,99895983.4,52,3851,30,1.921077e+06


Branch Performance Summary

In [45]:
if 'branch_id' in sales.columns:

    print("=" * 60)
    print("BRANCH PERFORMANCE")
    print("=" * 60)

    display(
        branch_kpi.head(10)
    )

BRANCH PERFORMANCE


,branch_id,total_sales,total_orders,total_quantity,total_customers,average_order_value
1,CHN001,6.615942e+09,4067,280446,102,1.626738e+06
4,KOL001,6.100494e+09,3794,256095,96,1.607932e+06
0,AHM001,5.616586e+09,3446,236347,88,1.629886e+06
2,DEL001,4.926515e+09,2953,204734,71,1.668308e+06
5,PUN001,4.766708e+09,2972,201749,75,1.603872e+06
3,HYD001,4.398510e+09,2768,189163,68,1.589057e+06


Final KPIs Summary

In [46]:
print("=" * 70)
print("WEEK 3 – FINAL KPI SUMMARY")
print("=" * 70)

display(kpi_df)

WEEK 3 – FINAL KPI SUMMARY


,KPI,Value
0,Total Sales,3.242475e+10
1,Total Orders,2.000000e+04
2,Total Quantity Sold,1.368534e+06
3,Total Products,3.000000e+01
4,Total Customers,5.000000e+02
5,Total Suppliers,8.000000e+00
6,Average Order Value,1.621238e+06
7,Average Quantity per Order,6.842670e+01
8,Total Purchase Value,3.681015e+12
9,Total Inventory Records,1.800000e+02


Final Validation

In [47]:
print("=" * 70)
print("WEEK 3 KPI ANALYSIS – COMPLETION CHECK")
print("=" * 70)

print(
    "✓ Sales KPIs calculated"
)

print(
    "✓ Product KPIs calculated"
)

print(
    "✓ Customer KPIs calculated"
)

print(
    "✓ Supplier KPI calculated"
)

print(
    "✓ Inventory KPIs calculated"
)

print(
    "✓ Stock health KPIs calculated"
)

print(
    "✓ Stock utilisation calculated"
)

print(
    "✓ Reorder gap calculated"
)

if 'branch_id' in sales.columns:
    print(
        "✓ Branch KPIs calculated"
    )

if 'sales_channel' in sales.columns:
    print(
        "✓ Sales channel KPIs calculated"
    )

print(
    "\nWeek 3 KPI analysis completed successfully."
)

WEEK 3 KPI ANALYSIS – COMPLETION CHECK
✓ Sales KPIs calculated
✓ Product KPIs calculated
✓ Customer KPIs calculated
✓ Supplier KPI calculated
✓ Inventory KPIs calculated
✓ Stock health KPIs calculated
✓ Stock utilisation calculated
✓ Reorder gap calculated
✓ Branch KPIs calculated
✓ Sales channel KPIs calculated

Week 3 KPI analysis completed successfully.
